# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a scoring task, not classification. We're not predicting yes/no; we're predicting a priority score for each page that ranks opportunity by likelihood of improvement if refreshed.
Why scoring? Because all 1,674 pages meet the base criteria (demand + weak traffic), but editors can't review all 1,674. ML ranks them by fixability.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: Refresh priority score, built from three signals:

Search volume (magnitude of opportunity)
Impressions gap (current_impressions vs. potential given volume)
Fixability (inferred from features):

High: page ranks for the keyword + high competition + weak position → fixable
Low: high volume + zero impressions → likely delisted/unfixable



The score is a proxy: pages with high scores are more likely to improve if refreshed.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**

"Of the top 50 pages we recommend to the editor, how many actually have high ROI potential?"

We care more about not wasting editor time than finding every opportunity.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

# Filter to your 1,674 pages:
# High search volume (top quartile) + suppressed traffic (bottom half impressions)
high_volume = df['search_volume'] > df['search_volume'].quantile(0.75)
low_impressions = df['impressions_90d'] < df['impressions_90d'].quantile(0.5)

opportunity_pages = df[high_volume & low_impressions].copy()

print(f"Total pages in dataset: {len(df)}")
print(f"Opportunity pages (high demand + weak traffic): {len(opportunity_pages)}")
print(f"\nOpportunity rate: {len(opportunity_pages) / len(df) * 100:.1f}%")

print("\n" + "="*100)
print("Sample of opportunity pages (unit of analysis: one row = one page):")
print("="*100)

display_cols = ['content_id', 'search_volume', 'impressions_90d', 'avg_position',
                'competition', 'ctr', 'engagement_rate', 'days_since_last_update']

print(opportunity_pages[display_cols].head(5))

print("\n" + "="*100)
print("Summary stats for opportunity pages:")
print("="*100)
print(opportunity_pages[['search_volume', 'impressions_90d', 'avg_position', 'competition', 'ctr']].describe())

Total pages in dataset: 30000
Opportunity pages (high demand + weak traffic): 3225

Opportunity rate: 10.8%

Sample of opportunity pages (unit of analysis: one row = one page):
              content_id  search_volume  impressions_90d  avg_position  \
19  content_af865035b328           30.0               99           6.9   
25  content_033ae3e7aecf           70.0               27           7.2   
28  content_19ad8f9bac29          480.0              315          59.3   
30  content_249298388b45          590.0              176          50.4   
39  content_4595e8704e07           90.0                4          36.3   

    competition   ctr  engagement_rate  days_since_last_update  
19         1.00  2.02              0.0                      20  
25         0.81  0.00              0.0                      20  
28         0.58  0.00              0.0                      20  
30         0.00  0.00              0.0                      22  
39         0.06  0.00              0.0               

**Unit of Analysis: One row = one page**

A page with:
- High search demand (e.g., 480/month)
- Weak 90-day traffic (315 impressions = underperforming)
- Measurable rank position (59.3 = ranking but far down)
- High competition (0.58 = competitive keyword)

These pages have *proven opportunity* (demand exists) but *poor placement* (fixable with content refresh).

Sample:

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule:

A simple rule would be:
"IF search_volume > 100 AND impressions_90d < 200
 AND avg_position > 10 THEN refresh this page"

But this fails because:

1. Position threshold is arbitrary — a page at position 8 might need refresh
   (if it has high volume + low impressions), but a page at position 45 might
   be unfixable (already delisted).

2. A page with 4 impressions + 500/month volume might be delisted, not just
   "needs refresh." A rule can't tell the difference.

3. Days since update matters, but how much? Old content (100 days) needs
   refresh, but what if it's already ranking well? A rule guesses; ML learns.

ML learns the tradeoffs from data. It finds which *combinations* of these
signals actually predicted pages that improved after refresh.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.